# 🧑‍🎤 Avatar Studio — Colab Renderer

Make a face **speak your generated voiceover** using a free Colab GPU.

**Workflow:**
1. In your local app's **Voice Studio**, type your script → **Generate Speech** → **Download MP3**.
2. Have a **photo** (jpg/png) or a short **face video** (mp4) ready.
3. Run the cells below: upload the audio + face, then pick **Option A** (photo) or **Option B** (video).
4. Preview and **download** the talking-avatar MP4.

> Run **Option A *or* Option B per session** — they use conflicting package versions. To switch engines, use *Runtime → Restart runtime* first.

## Step 0 — Confirm the GPU is on
First enable it: **Runtime → Change runtime type → Hardware accelerator: GPU → Save**, then run:

In [ ]:
import subprocess
r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True)
print(r.stdout.strip() if r.returncode == 0 else 'No GPU detected — Runtime > Change runtime type > GPU, then re-run.')

## Step 1 — Upload your audio + face
Upload the **MP3 from Voice Studio**, then your **photo or video**. Their paths are stored in `AUDIO` and `FACE`.

In [ ]:
from google.colab import files

print('⬆️  Upload your AUDIO file (the .mp3 from Voice Studio):')
_a = files.upload()
AUDIO = '/content/' + list(_a.keys())[0]

print('\n⬆️  Upload your FACE — a PHOTO (jpg/png) or VIDEO (mp4):')
_f = files.upload()
FACE = '/content/' + list(_f.keys())[0]

print('\nAUDIO =', AUDIO)
print('FACE  =', FACE)

---
## Option A — Photo Avatar (SadTalker)
Best when you only have **one still photo**. Animates head + lips from the image.

### A1. Install SadTalker + download models (~3–5 min)

In [ ]:
%cd /content
!git clone https://github.com/OpenTalker/SadTalker
%cd /content/SadTalker
!pip install -q -r requirements.txt
# Download all pretrained checkpoints (SadTalker + GFPGAN enhancer)
!bash scripts/download_models.sh

In [ ]:
# Common Colab fix: newer torchvision removed `functional_tensor`, which basicsr/gfpgan import.
# This patch rewrites that import so the GFPGAN enhancer loads. Safe to run even if not needed.
import glob
patched = False
for p in glob.glob('/usr/local/lib/python3*/dist-packages/basicsr/data/degradations.py'):
    s = open(p).read()
    if 'functional_tensor' in s:
        open(p, 'w').write(s.replace('functional_tensor', 'functional'))
        print('patched', p); patched = True
print('done' if patched else 'nothing to patch (already fine)')

In [ ]:
# A2. Render. --still = less head motion (good for portraits); --enhancer gfpgan = sharper face.
%cd /content/SadTalker
!python inference.py \
  --driven_audio {AUDIO} \
  --source_image {FACE} \
  --result_dir /content/results \
  --still --preprocess full --enhancer gfpgan

In [ ]:
# A3. Find newest result, preview, and download.
import glob, os
from IPython.display import HTML
from base64 import b64encode
from google.colab import files

vids = sorted(glob.glob('/content/results/**/*.mp4', recursive=True), key=os.path.getmtime)
assert vids, 'No output video found — check the log above for errors.'
OUT = vids[-1]
print('Result:', OUT)

data = b64encode(open(OUT, 'rb').read()).decode()
display(HTML(f'<video width=512 controls src="data:video/mp4;base64,{data}"></video>'))
files.download(OUT)

---
## Option B — Lip Sync (Wav2Lip)
Best when you have a **face video** (also works on a still photo). Lighter and faster.

> If you already ran Option A, do **Runtime → Restart runtime** first, then re-run Step 1 to re-upload, then run these.

### B1. Install Wav2Lip (Colab-friendly fork) + checkpoints

In [ ]:
%cd /content
!git clone https://github.com/justinjohn0306/Wav2Lip
%cd /content/Wav2Lip

# The fork pins batch-face==1.5.0.dev0, which was removed from PyPI. Install a
# current batch-face, then the remaining requirements without that broken pin.
!pip install -q batch-face
!sed -i '/batch.face/d' requirements.txt
!pip install -q -r requirements.txt

# Model weights (justinjohn0306 'models' release). If any link 404s, grab the
# current URL from the repo's "Getting the weights" table and replace below.
B = 'https://github.com/justinjohn0306/Wav2Lip/releases/download/models'
!wget -q "$B/wav2lip.pth"      -O checkpoints/wav2lip.pth
!wget -q "$B/wav2lip_gan.pth"  -O checkpoints/wav2lip_gan.pth
!wget -q "$B/s3fd.pth"         -O face_detection/detection/sfd/s3fd.pth
print('checkpoints downloaded')

In [ ]:
# B2. Render. wav2lip_gan.pth = sharper; --nosmooth helps with quick mouth movements.
%cd /content/Wav2Lip
!python inference.py \
  --checkpoint_path checkpoints/wav2lip_gan.pth \
  --face {FACE} \
  --audio {AUDIO} \
  --outfile /content/result_wav2lip.mp4 \
  --pads 0 10 0 0 --nosmooth

In [ ]:
# B3. Preview + download.
from IPython.display import HTML
from base64 import b64encode
from google.colab import files

OUT = '/content/result_wav2lip.mp4'
data = b64encode(open(OUT, 'rb').read()).decode()
display(HTML(f'<video width=512 controls src="data:video/mp4;base64,{data}"></video>'))
files.download(OUT)

---
## Tips & troubleshooting
- **`batch-face` install error:** already handled in B1 (it installs a current build and strips the broken pin). If it still complains, run `!pip install batch-face` in a new cell, then re-run B2.
- **Out of memory / crash:** shrink your photo/video (try a 512px image) or add `--resize_factor 2` to the Wav2Lip command.
- **Face not detected (Wav2Lip):** the face must be reasonably large and front-facing. Try `--pads 0 20 0 0`.
- **SadTalker `functional_tensor` error:** re-run the patch cell (A1's second cell), then re-run A2.
- **Robotic / clipped lips:** regenerate the audio in Voice Studio with a slightly slower **Speed**.
- **Checkpoint 404 (Wav2Lip):** open <https://github.com/justinjohn0306/Wav2Lip> → "Getting the weights" and paste the current links into B1.
- **Quality vs speed (SadTalker):** drop `--enhancer gfpgan` for a faster, softer result; use `--preprocess crop` to zoom on the face.